In [ ]:
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
import torch
import json

In [ ]:
model_name = 'openai-community/gpt2'

In [ ]:
model = GPT2LMHeadModel.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
model

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [ ]:
tokenizer = GPT2TokenizerFast.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
len(tokenizer)

50257

In [ ]:
promt = 'Hi'

In [ ]:
tokenizer.encode(promt)

[17250]

In [ ]:
tokenizer(promt)

{'input_ids': [17250], 'attention_mask': [1]}

In [ ]:
tokenizer(promt, return_tensors='pt')

{'input_ids': tensor([[17250]]), 'attention_mask': tensor([[1]])}

In [ ]:
model(input_ids=torch.tensor([[17250]]))

CausalLMOutputWithCrossAttentions(loss=None, logits=tensor([[[-34.2418, -34.3303, -37.3033,  ..., -43.0448, -42.7169, -35.2205]]],
       grad_fn=<UnsafeViewBackward0>), past_key_values=DynamicCache(layers=[DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer]), hidden_states=None, attentions=None, cross_attentions=None)

In [ ]:
enc = tokenizer(promt, return_tensors='pt')
enc

{'input_ids': tensor([[17250]]), 'attention_mask': tensor([[1]])}

In [ ]:
output = model(**enc)
output

CausalLMOutputWithCrossAttentions(loss=None, logits=tensor([[[-34.2418, -34.3303, -37.3033,  ..., -43.0448, -42.7169, -35.2205]]],
       grad_fn=<UnsafeViewBackward0>), past_key_values=DynamicCache(layers=[DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer]), hidden_states=None, attentions=None, cross_attentions=None)

In [ ]:
output.logits

tensor([[[-34.2418, -34.3303, -37.3033,  ..., -43.0448, -42.7169, -35.2205]]],
       grad_fn=<UnsafeViewBackward0>)

In [ ]:
output.logits.shape

torch.Size([1, 1, 50257])

In [ ]:
output.logits.squeeze()

tensor([-34.2418, -34.3303, -37.3033,  ..., -43.0448, -42.7169, -35.2205],
       grad_fn=<SqueezeBackward0>)

In [ ]:
output.logits.squeeze().argmax()

tensor(13)

In [ ]:
output.logits[0][-1].argmax()

tensor(13)

In [ ]:
tokenizer.decode(torch.tensor(13))

'.'

In [ ]:
out = model.generate(enc['input_ids'], max_length=100)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [ ]:
out

tensor([[17250,    13,   314,  1101,  7926,    11,   475,   314,  1101,   407,
          1654,   611,   345,   821,  3910,   286,   428,    13,   314,  1101,
           407]])

In [ ]:
tokenizer.decode(out[0])

"Hi. I'm sorry, but I'm not sure if you're aware of this. I'm not sure if you're aware of this.\n\nI'm sorry, but I'm not sure if you're aware of this. I'm not sure if you're aware of this.\n\nI'm sorry, but I'm not sure if you're aware of this. I'm not sure if you're aware of this.\n\nI'm sorry, but I'm not sure if you"

In [ ]:
promt = 'Who is Elon Mask'

In [ ]:
enc = tokenizer(promt, return_tensors='pt')
enc

{'input_ids': tensor([[ 8241,   318, 32451, 18007]]), 'attention_mask': tensor([[1, 1, 1, 1]])}

In [ ]:
out = model.generate(enc['input_ids'], max_length=50)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [ ]:
tokenizer.decode(out[0])

'Who is Elon Masked?"\n\n"I\'m Elon Masked," he said. "I\'m a guy who\'s been in the business for a long time. I\'m a guy who\'s been in the business for a long time. I'

In [ ]:
data = json.load(open('instruction-data.json'))

In [ ]:
data[0]

{'instruction': 'Evaluate the following phrase by transforming it into the spelling given.',
 'input': 'freind --> friend',
 'output': 'The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".'}

In [ ]:
len(data)

1100